In [1]:
%pip install sentence-transformers pinecone-client openai pdfplumber rank_bm25 tiktoken


   ---------------------------------------- 0.0/5.6 MB ? eta -:--:--
   ----------- ---------------------------- 1.6/5.6 MB 7.6 MB/s eta 0:00:01
   -------------- ------------------------- 2.1/5.6 MB 7.8 MB/s eta 0:00:01
   -------------- ------------------------- 2.1/5.6 MB 7.8 MB/s eta 0:00:01
   ---------------- ----------------------- 2.4/5.6 MB 3.4 MB/s eta 0:00:01
   ---------------- ----------------------- 2.4/5.6 MB 3.4 MB/s eta 0:00:01
   -------------------------- ------------- 3.7/5.6 MB 2.9 MB/s eta 0:00:01
   ------------------------------------- -- 5.2/5.6 MB 3.6 MB/s eta 0:00:01
   ---------------------------------------- 5.6/5.6 MB 3.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/884.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/884.3 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/884.3 kB ? eta -:--:--
   ---------------------------------------- 884.3/884.3 kB 8.0 MB/s eta 0:00:00
   -------------------------


[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, json, hashlib
import pdfplumber
from rank_bm25 import BM25Okapi
import pinecone

USE_OPENAI = False  # switch True/False

if USE_OPENAI:
    import openai
    openai.api_key = os.getenv("OPENAI_API_KEY")
    EMBED_DIM = 1536
    def get_embeddings(texts):
        resp = openai.Embedding.create(model="text-embedding-3-small", input=texts)
        return [d["embedding"] for d in resp["data"]]
else:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("all-MiniLM-L6-v2")
    EMBED_DIM = 384
    def get_embeddings(texts):
        return model.encode(texts, normalize_embeddings=True).tolist()


c:\Users\Anjali Sinha\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [59]:
def extract_text_from_pdf(path):
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() + "\n"
    return text

pdf_path = "Bhagavad-gita.pdf"
raw_text = extract_text_from_pdf(pdf_path)


Task exception was never retrieved
future: <Task finished name='Task-1' coro=<Server.serve() done, defined at c:\Users\Anjali Sinha\AppData\Local\Programs\Python\Python312\Lib\site-packages\uvicorn\server.py:69> exception=KeyboardInterrupt()>
Traceback (most recent call last):
  File "c:\Users\Anjali Sinha\AppData\Local\Programs\Python\Python312\Lib\site-packages\uvicorn\main.py", line 580, in run
    server.run()
  File "c:\Users\Anjali Sinha\AppData\Local\Programs\Python\Python312\Lib\site-packages\uvicorn\server.py", line 67, in run
    return asyncio.run(self.serve(sockets=sockets))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Anjali Sinha\AppData\Roaming\Python\Python312\site-packages\nest_asyncio.py", line 30, in run
    return loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Anjali Sinha\AppData\Roaming\Python\Python312\site-packages\nest_asyncio.py", line 92, in run_until_complete
    self._run_once()
  File "C:\Use

In [60]:
def chunk_text(text, size=500, overlap=50):
    words, chunks, i = text.split(), [], 0
    while i < len(words):
        chunks.append(" ".join(words[i:i+size]))
        i += size - overlap
    return chunks

chunks = chunk_text(raw_text)


In [61]:
pc.list_indexes()



[
    {
        "name": "hybrid-search-langchain-pinecone",
        "metric": "dotproduct",
        "host": "hybrid-search-langchain-pinecone-ndhnkhy.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 384,
        "deletion_protection": "disabled",
        "tags": null
    },
    {
        "name": "bhagavad-gita-hybrid",
        "metric": "cosine",
        "host": "bhagavad-gita-hybrid-ndhnkhy.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 384,
        "deletion_pro

In [62]:
EMBED_DIM = 384  # if using Hugging Face all-MiniLM-L6-v2

pc.create_index(
    name="bhagavad-gita-hybrid",
    dimension=EMBED_DIM,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",       # check Pinecone dashboard (can be "gcp" too)
        region="us-east-1" # replace with your region
    )
)


PineconeApiException: (409)
Reason: Conflict
HTTP response headers: HTTPHeaderDict({'content-type': 'text/plain; charset=utf-8', 'access-control-allow-origin': '*', 'vary': 'origin,access-control-request-method,access-control-request-headers', 'access-control-expose-headers': '*', 'x-pinecone-api-version': '2025-04', 'x-cloud-trace-context': '5742cfdb3ac5482abd84f7e54201057d', 'date': 'Mon, 15 Sep 2025 16:50:43 GMT', 'server': 'Google Frontend', 'Content-Length': '85', 'Via': '1.1 google', 'Alt-Svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000'})
HTTP response body: {"error":{"code":"ALREADY_EXISTS","message":"Resource  already exists"},"status":409}


In [63]:
import os
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key="pcsk_VM5Mr_Rj2DSWHQtifPXCTWUrfDMg436KtvaFvsnpW4AzRywe3BMDtLeqX35BCQG3KguP2")


In [64]:
CACHE_FILE = "embeddings_cache.json"
cache = json.load(open(CACHE_FILE)) if os.path.exists(CACHE_FILE) else {}

def cached_embedding(text):
    key = hashlib.md5(text.encode()).hexdigest()
    if key in cache: return cache[key]
    emb = get_embeddings([text])[0]
    cache[key] = emb
    json.dump(cache, open(CACHE_FILE,"w"))
    return emb


In [65]:
index = pc.Index("bhagavad-gita-hybrid")


In [67]:
vectors = [
    (f"chunk-{i}", cached_embedding(c), {"text": c})
    for i, c in enumerate(chunks)
]

index.upsert(vectors=vectors)


{'upserted_count': 304}

In [69]:
bm25 = BM25Okapi([c.split() for c in chunks])


In [70]:
def hybrid_search(query, top_k=5, alpha=0.7):
    q_emb = get_embeddings([query])[0]
    pine_results = index.query(vector=q_emb, top_k=top_k*2, include_metadata=True)
    pine_dict = {m["id"]: m["score"] for m in pine_results["matches"]}
    bm25_scores = bm25.get_scores(query.split())

    combined = []
    for i, c in enumerate(chunks):
        score = alpha * pine_dict.get(f"chunk-{i}",0) + (1-alpha)*bm25_scores[i]
        combined.append((score, c))
    return sorted(combined,key=lambda x:x[0],reverse=True)[:top_k]


In [71]:
results = hybrid_search("What is the essence of karma yoga?")
for s, c in results:
    print(f"Score: {s:.4f}\n{c[:200]}...\n{'-'*50}")


Score: 4.5932
mind of a serious student of yoga. K®ß√a has made it very clear that He is eternal, without birth, supreme and imper- ishable. He knows past, present and future; He knows all living beings and those w...
--------------------------------------------------
Score: 4.4507
military leader. Thus one is encouraged to follow that line of work. When one performs the work prescribed to him accord- ing to his qualities, and devotes that work to please the Supreme Person K®ß√a...
--------------------------------------------------
Score: 4.2416
इित मे मितः । वयाना त यतता शोऽवामपायतः ॥३६॥ asaµyatåtmanå yogo dußpråpa iti me mati˙ vaçyåtmanå tu yatatå çakyo’våptum upåyata˙ My conclusion is that yoga is difficult to attain if one’s mind i...
--------------------------------------------------
Score: 3.8639
िववते योगं ोवानहमयम ् । िववानवे ाह मनिराकवेऽवीत ् ॥१॥ çrî bhagavån uvåca – imaµ vivasvate yogaµ proktavån aham avyayam vivasvån manave pråha manur ikßvåkave’bravît Bhaga

In [72]:
results = hybrid_search("What is krishna like?")
for s, c in results:
    print(f"Score: {s:.4f}\n{c[:200]}...\n{'-'*50}")

Score: 2.5724
mind of a serious student of yoga. K®ß√a has made it very clear that He is eternal, without birth, supreme and imper- ishable. He knows past, present and future; He knows all living beings and those w...
--------------------------------------------------
Score: 2.2113
military leader. Thus one is encouraged to follow that line of work. When one performs the work prescribed to him accord- ing to his qualities, and devotes that work to please the Supreme Person K®ß√a...
--------------------------------------------------
Score: 2.1525
after many lifetimes – he attains the Supreme Destination. VERSE 46 तपिोऽिधको योगी ािनोऽिप मतोऽिधकः । किमािधको योगी ताोगी भवाजन ॥४६॥ tapasvibhyo’dhiko yogî jñånibhyo’pi mato’dhika˙ karmibhy...
--------------------------------------------------
Score: 1.8040
primeval because it is the oldest of all. It is imperceptible because it lies beyond the range of the physical senses. It is inconceivable because it is beyond the speculative fun

In [73]:
results = hybrid_search("Who is won the war?")
for s, c in results:
    print(f"Score: {s:.4f}\n{c[:200]}...\n{'-'*50}")

Score: 3.2869
mind of a serious student of yoga. K®ß√a has made it very clear that He is eternal, without birth, supreme and imper- ishable. He knows past, present and future; He knows all living beings and those w...
--------------------------------------------------
Score: 2.9692
tåmasî O Pårtha, the determination of those who cannot con- quer sleep, fear, lamentation, misery and pride is in the mode of ignorance. Anuv®tti Throughout the Bhagavad-gîtå, Çrî K®ß√a addresses Arju...
--------------------------------------------------
Score: 2.9657
his own sons, led by the corrupt Duryodhana, would ascend the imperial throne. To this end, and with the consent of his father, Duryodhana made several assassination attempts upon the lives of the På√...
--------------------------------------------------
Score: 2.8131
the mahå-mantra, the Kali-santara√a Upanißad states as follows: hare k®ß√a hare k®ß√a k®ß√a k®ß√a hare hare hare råma hare råma råma råma hare hare iti ßo∂açakaµ nåmnåµ kali-kalma

In [74]:
query_vector = model.encode(["What does the Gita say about self-realisation?"])[0]


In [76]:
# Step 1: Write your question
question = "What does the Gita say about detachment?"

# Step 2: Convert question to embedding
query_vector = model.encode([question])[0].tolist()

# Step 3: Query Pinecone
results = index.query(
    vector=query_vector,
    top_k=3,
    include_metadata=True
)

# Step 4: Print top answers
for match in results['matches']:
    print(f"Score: {match['score']}")
    print(match['metadata']['text'])
    print("------")


Score: 0.509467185
to Arjuna in Bhagavad-gîtå and thus one who studies the Gîtå hears from K®ß√a directly. The philosophy of Bhagavad-gîtå is clear for the sincere reader, yet for some, approaching the Gîtå may seem daunt- ing – its language too ancient. However, this obstacle is easily overcome by a straightforward translation and com- mentary (Anuv®tti). The requirement for a translation and commentary on the Gîtå is as necessary today as anytime in the past. With the passing of time, our values and our world view are constantly changing, and this demands a fresh approach to the understanding of the Gîtå. This translation and commentary on the Bhagavad-gîtå provides simple, yet profound knowledge to elevate us to a higher state of consciousness whereby we can realise our ix true self and progress towards attaining a life of spiritual fulfilment. Self-realisation means to realise our actual purpose in life and act towards it, gradually freeing us from the yoke of material bondage. Whe

In [78]:
def ask_gita(question, top_k=3):
    # 1. Convert question into embedding
    query_vector = model.encode([question])[0]
    
    # 2. Query Pinecone
    results = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True
    )
    
    # 3. Return the best answer (first match)
    best_match = results['matches'][0]['metadata']['text']
    return best_match


In [80]:
print(index.describe_index_stats())



{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 304}},
 'total_vector_count': 304,
 'vector_type': 'dense'}


In [81]:
print(len(query_vector))


384


In [83]:
def ask_gita(question, top_k=3):
    # Encode the question
    query_vector = model.encode([question])[0]

    # Query Pinecone
    results = index.query(
        vector=[query_vector],   # must be a list
        top_k=top_k,
        include_metadata=True
    )

    # Return best match
    best_match = results['matches'][0]['metadata']['text']
    return best_match


In [85]:
from pinecone import Pinecone

pc = Pinecone(api_key="pcsk_VM5Mr_Rj2DSWHQtifPXCTWUrfDMg436KtvaFvsnpW4AzRywe3BMDtLeqX35BCQG3KguP2")
index = pc.Index("bhagavad-gita-hybrid")

results = index.query(
    vector=[query_vector],
    top_k=3,
    include_metadata=True
)


In [86]:
def ask_gita(question, top_k=3):
    # Step 1: Encode the question
    query_vector = model.encode([question])[0]

    # Step 2: Query Pinecone
    results = index.query(
        vector=[query_vector],    # MUST be inside a list
        top_k=top_k,
        include_metadata=True
    )

    # Step 3: Return the best match
    best_match = results['matches'][0]['metadata']['text']
    return best_match


In [88]:
results = hybrid_search("What is the essence of karma yoga?")
for s, c in results:
    print(f"Score: {s:.4f}\n{c[:200]}...\n{'-'*50}")


Score: 4.5932
mind of a serious student of yoga. K®ß√a has made it very clear that He is eternal, without birth, supreme and imper- ishable. He knows past, present and future; He knows all living beings and those w...
--------------------------------------------------
Score: 4.4507
military leader. Thus one is encouraged to follow that line of work. When one performs the work prescribed to him accord- ing to his qualities, and devotes that work to please the Supreme Person K®ß√a...
--------------------------------------------------
Score: 4.2416
इित मे मितः । वयाना त यतता शोऽवामपायतः ॥३६॥ asaµyatåtmanå yogo dußpråpa iti me mati˙ vaçyåtmanå tu yatatå çakyo’våptum upåyata˙ My conclusion is that yoga is difficult to attain if one’s mind i...
--------------------------------------------------
Score: 3.8639
िववते योगं ोवानहमयम ् । िववानवे ाह मनिराकवेऽवीत ् ॥१॥ çrî bhagavån uvåca – imaµ vivasvate yogaµ proktavån aham avyayam vivasvån manave pråha manur ikßvåkave’bravît Bhaga

In [89]:
def ask_gita(question, top_k=3):
    results = hybrid_search(question)
    # just return the best one
    return results[0][1]


In [90]:
print(ask_gita("What does the Gita say about self-realisation?"))


mind of a serious student of yoga. K®ß√a has made it very clear that He is eternal, without birth, supreme and imper- ishable. He knows past, present and future; He knows all living beings and those who fix their minds in meditation on K®ß√a, knowing Him to be the Controller of everything, will not have to take birth again in this material world. ॐ तिदित ीमहाभारते शतसाहां संिहतायां वैयािसां भीपवािण ीमगवीतासूपिनष  िवायां योगशाे ीकाजनसंवादे ानिवानयोगो नाम समोऽायः॥ oµ tat saditi çrî-mahåbhårate-çata-såhasryåµ saµhitåyåµ vaiyåsikyåµ bhîßma-parvå√i çrîmad bhagavad-gîtåsüpanißatsu brahma-vidyåyåµ yoga-çåstre çrî k®ß√årjuna-saµvåde jñåna-vijñåna yogo nåma saptamo’dhyåya˙ OÂ TAT SAT – Thus ends Chapter Seven entitled Jñåna- Vijñåna Yoga from the conversation between Çrî K®ß√a and Arjuna in the Upanißad known as Çrîmad Bhagavad-gîtå, the yoga-çåstra of divine knowledge, from the Bhîßma- parva of Mahåbhårata, the literature revealed by Vyåsa in one hundred thousand verses

In [97]:
# backend.py
from fastapi import FastAPI
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()

# Allow React frontend to call this API
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # replace * with your frontend URL in production
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class QueryRequest(BaseModel):
    question: str

@app.post("/ask")
def ask_gita_api(req: QueryRequest):
    # use your existing function
    answer = ask_gita(req.question)
    return {"answer": answer}


In [95]:
from pyngrok import ngrok, conf

ngrok.set_auth_token("2hx6OHjuyXJKF3yulOLo1aaVmqS_29vC9nnyyDWgnc3bcVQCi")




In [100]:
def ask_gita(question):
    response = model.generate(question, max_tokens=100)  # limits output length
    return response


In [103]:
# main.py
from fastapi import FastAPI
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class QueryRequest(BaseModel):
    question: str

@app.post("/ask")
def ask_gita_api(req: QueryRequest):
    answer = ask_gita(req.question)   # your function
    return {"answer": answer}


Task exception was never retrieved
future: <Task finished name='Task-52' coro=<Server.serve() done, defined at c:\Users\Anjali Sinha\AppData\Local\Programs\Python\Python312\Lib\site-packages\uvicorn\server.py:69> exception=KeyboardInterrupt()>
Traceback (most recent call last):
  File "c:\Users\Anjali Sinha\AppData\Local\Programs\Python\Python312\Lib\site-packages\uvicorn\main.py", line 580, in run
    server.run()
  File "c:\Users\Anjali Sinha\AppData\Local\Programs\Python\Python312\Lib\site-packages\uvicorn\server.py", line 67, in run
    return asyncio.run(self.serve(sockets=sockets))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Anjali Sinha\AppData\Roaming\Python\Python312\site-packages\nest_asyncio.py", line 30, in run
    return loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Anjali Sinha\AppData\Roaming\Python\Python312\site-packages\nest_asyncio.py", line 92, in run_until_complete
    self._run_once()
  File "C:\Us

In [106]:
from fastapi import FastAPI
app = FastAPI()


In [2]:
def truncate_answer(answer: str, min_words=40, max_words=65) -> str:
    words = answer.split()
    if len(words) > max_words:
        return " ".join(words[:max_words]) + "..."
    elif len(words) < min_words:
        return answer + " (short answer)"
    return answer
